# Week 1 — HPC Foundations, Hardware Architectures, and Profiling

**Course:** High-Performance Computing & Scaling Large Models  
**Instructor:** Haydar Kılıç  
**Repository:** [HAYDARKILIC/hpc-scaling-llms](https://github.com/HAYDARKILIC/hpc-scaling-llms)

---

## Learning Objectives

By the end of this lecture, students will be able to:

1. Articulate the *memory hierarchy* of modern GPUs and explain why bandwidth, not FLOPs, is often the binding constraint in deep learning workloads.
2. Apply the **roofline model** to classify a kernel as compute-bound or memory-bound.
3. Profile a PyTorch model using the built-in `torch.profiler` and interpret the resulting trace.
4. Use **NVIDIA Nsight Systems** and **Nsight Compute** at a conceptual level to diagnose performance bottlenecks.

## Prerequisites

- Working knowledge of Python and PyTorch.
- Access to an NVIDIA GPU with CUDA 12.1 or later.
- Familiarity with linear algebra and basic deep learning concepts.


## 1. From Moore's Law to the Memory Wall

For four decades, the performance of computational systems was governed by an empirical regularity: every 18–24 months, transistor density doubled, and with it the achievable single-thread performance of general-purpose CPUs. This regime — *Dennard scaling* — collapsed around 2005 as power density saturated. The industry responded by replicating cores rather than accelerating them, and later by introducing *specialized accelerators* such as the GPU, the TPU, and a growing constellation of domain-specific architectures.

The training of large language models inherits the consequences of this transition in a particularly acute form. A single forward-backward pass over a transformer with $N$ parameters performs on the order of $6 N \cdot B \cdot L$ floating-point operations, where $B$ is the batch size and $L$ the sequence length. For a 70-billion-parameter model with $B = 1, L = 2048$, this is approximately $0.86$ TFLOPs *per token*. A modern A100 GPU is rated at 312 TFLOPs/s in FP16 — yet practitioners routinely observe sustained throughput between 30 % and 50 % of peak. The gap is explained almost entirely by **memory bandwidth**.

### The Three Bottlenecks

Every operation executed on a GPU is constrained by one of three resources:

| Bottleneck | Symptom | Example |
|------------|---------|---------|
| **Compute-bound** | High FLOPs utilization, low memory traffic per FLOP | Large `GEMM` (matrix–matrix multiply) |
| **Memory-bound** | Low FLOPs utilization, traffic saturates HBM bandwidth | Activation functions, layer normalization, attention softmax |
| **Communication-bound** | GPUs idle waiting for `all-reduce` or `all-gather` | Distributed data parallelism with small effective batches |

We will revisit each of these in the weeks ahead.


## 2. Anatomy of an NVIDIA Datacenter GPU

The diagram below summarizes the salient features of the NVIDIA A100 (Ampere, GA100) at three levels of abstraction:

```
┌──────────────────────────────────────────────────────────────┐
│                       NVIDIA A100 (SXM4)                     │
│                                                              │
│   108 × Streaming Multiprocessors (SMs)                      │
│   Each SM: 64 FP32 + 64 INT32 + 4 Tensor Cores               │
│                                                              │
│   Register file (per SM): 256 KB                             │
│   L1 / Shared memory (per SM): 192 KB (configurable)         │
│   L2 cache (chip-wide): 40 MB                                │
│   HBM2e: 40 / 80 GB at 1555 / 2039 GB/s                      │
│                                                              │
│   NVLink 3.0: 600 GB/s GPU↔GPU                               │
│   PCIe 4.0: 64 GB/s host↔device                              │
└──────────────────────────────────────────────────────────────┘
```

Several quantitative facts deserve emphasis:

- The L2 cache (40 MB) is two orders of magnitude smaller than typical activations of a transformer layer. **Most data movement is to and from HBM**, not from on-chip caches.
- HBM2e bandwidth — 2.0 TB/s on the 80 GB SKU — is roughly **20× higher** than PCIe 4.0 (64 GB/s). This justifies investing engineering effort to keep data resident on the GPU.
- Tensor Cores accept *tiles* (typically $16 \times 16$ or $16 \times 8$). Their peak throughput is achieved only when both operands are in shared memory or registers; spilling to global memory destroys the win.

### The Hopper Generation (H100, H200)

The Hopper architecture (2022) introduces:

- **FP8 Tensor Cores** with 4× the throughput of FP16 (1979 TFLOPs/s in dense FP8).
- **Thread Block Cluster** abstraction enabling cooperation across SMs.
- **TMA (Tensor Memory Accelerator)** for asynchronous global → shared memory copies.
- **DPX instructions** for dynamic programming kernels.

These features will become central in Week 3 (FlashAttention) and Week 5 (Megatron-LM optimizations).


## 3. The Roofline Model

The roofline model, introduced by Williams, Waterman, and Patterson (2009), provides a unifying visual framework for performance analysis. For a given kernel, we plot:

- **x-axis**: *arithmetic intensity* $I = \dfrac{\text{FLOPs}}{\text{bytes moved}}$, in FLOPs/byte.
- **y-axis**: *attainable performance* in FLOPs/s.

The attainable performance is bounded by

$$
P_\text{attainable}(I) = \min\bigl(P_\text{peak},\ \ \beta \cdot I\bigr)
$$

where $P_\text{peak}$ is the device's peak compute throughput and $\beta$ is its peak memory bandwidth. The two regimes meet at the **machine balance point** $I^* = P_\text{peak} / \beta$.

For an A100 in FP16:
- $P_\text{peak} = 312$ TFLOPs/s
- $\beta = 2.0$ TB/s
- $I^* = 156$ FLOPs/byte

Any kernel with arithmetic intensity below 156 FLOPs/byte is memory-bound on this device. Let us see this concretely.


In [ ]:
# Setup: imports and environment check
import os
import time
import math
import numpy as np
import torch
import torch.nn as nn

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version     : {torch.version.cuda}")
    print(f"GPU device       : {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Compute capab.   : {props.major}.{props.minor}")
    print(f"Total memory     : {props.total_memory / 1e9:.2f} GB")
    print(f"# SMs            : {props.multi_processor_count}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)


In [ ]:
# Roofline visualization for the A100 (FP16)
import matplotlib.pyplot as plt

PEAK_FLOPS = 312e12          # 312 TFLOPs/s in FP16 (Tensor Cores)
PEAK_BW    = 2.0e12          # 2.0 TB/s HBM2e
balance    = PEAK_FLOPS / PEAK_BW   # ≈ 156 FLOPs/byte

intensities = np.logspace(-1, 4, 256)
attainable  = np.minimum(PEAK_FLOPS, PEAK_BW * intensities)

# Reference kernels (approximate intensities)
kernels = {
    "Element-wise (ReLU)"     : 0.25,
    "LayerNorm"               : 1.5,
    "Attention softmax (naive)": 4.0,
    "GEMV (small batch)"      : 2.0,
    "GEMM 4096×4096"          : 256.0,
    "Conv 3×3, 256 ch"        : 80.0,
}

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(intensities, attainable / 1e12, lw=2, color="black")
ax.axvline(balance, ls="--", color="gray", alpha=0.7,
           label=f"Machine balance  Iˣ ≈ {balance:.0f} FLOPs/byte")

for name, I in kernels.items():
    perf = min(PEAK_FLOPS, PEAK_BW * I) / 1e12
    ax.scatter(I, perf, s=70, zorder=5)
    ax.annotate(name, (I, perf), xytext=(6, 6), textcoords="offset points",
                fontsize=9)

ax.set_xlabel("Arithmetic intensity  (FLOPs / byte)")
ax.set_ylabel("Attainable performance  (TFLOPs/s)")
ax.set_title("Roofline Model — NVIDIA A100 (FP16)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


**Reading the chart.** A kernel that lies on the *diagonal* portion of the curve is memory-bound: increasing its arithmetic intensity (e.g., by *fusing* operations to reuse data already in registers) will translate directly into higher throughput. A kernel on the *horizontal* portion has saturated the compute pipeline; only a different algorithm — or a different hardware generation — can improve it.

Notice that LayerNorm, ReLU, and softmax operate deep in the memory-bound regime. This is the central observation that motivates **kernel fusion** (Week 2), **FlashAttention** (Week 3), and a great deal of modern systems research.


## 4. Profiling with `torch.profiler`

PyTorch ships with a lightweight, tracing profiler that captures both Python-level operator invocations and the underlying CUDA kernels. We will profile a small transformer block and inspect where the time is spent.


In [ ]:
# A minimal transformer block — small enough to run on any modern GPU
class TransformerBlock(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048, dropout=0.0):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout,
                                          batch_first=True)
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        a, _ = self.attn(x, x, x, need_weights=False)
        x    = self.ln1(x + a)
        x    = self.ln2(x + self.ffn(x))
        return x


B, L, D = 8, 512, 512
model   = TransformerBlock(d_model=D).to(device).eval()
x       = torch.randn(B, L, D, device=device)

# Warmup
for _ in range(3):
    with torch.no_grad():
        _ = model(x)
torch.cuda.synchronize() if device.type == "cuda" else None

print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")


In [ ]:
# Profile the transformer block
from torch.profiler import profile, ProfilerActivity, record_function

activities = [ProfilerActivity.CPU]
if device.type == "cuda":
    activities.append(ProfilerActivity.CUDA)

with profile(activities=activities, record_shapes=True, profile_memory=True) as prof:
    with record_function("transformer_forward"):
        with torch.no_grad():
            for _ in range(10):
                _ = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()

sort_key = "cuda_time_total" if device.type == "cuda" else "cpu_time_total"
print(prof.key_averages().table(sort_by=sort_key, row_limit=15))


**Interpreting the table.** The profiler reports, for each operator:

- *Self CPU/CUDA time* — time spent inside the operator excluding nested calls.
- *# of calls* — number of invocations.
- *CUDA memory usage* — peak allocator delta (negative values denote releases).

In the output above you will typically see `aten::addmm` (the underlying GEMM behind `nn.Linear`) consuming the majority of CUDA time. The attention path — `aten::scaled_dot_product_attention`, `aten::softmax`, `aten::bmm` — is broken into many smaller kernels that, in aggregate, are often *memory-bound*. This is exactly the pathology FlashAttention addresses.


## 5. Roofline Analysis on a Live Kernel

We can estimate the *actual* arithmetic intensity of a GEMM operation by counting FLOPs and bytes moved.


In [ ]:
# Empirical measurement of GEMM performance at varying sizes
def benchmark_gemm(N, dtype=torch.float16, n_iter=50, warmup=10):
    if device.type != "cuda":
        return None
    a = torch.randn(N, N, device=device, dtype=dtype)
    b = torch.randn(N, N, device=device, dtype=dtype)

    for _ in range(warmup):
        c = a @ b
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(n_iter):
        c = a @ b
    end.record()
    torch.cuda.synchronize()

    elapsed_s   = start.elapsed_time(end) / 1000 / n_iter
    flops       = 2 * N**3                      # 2N³ for an N×N GEMM
    bytes_moved = 3 * N * N * a.element_size()  # A, B, C
    tflops      = flops / elapsed_s / 1e12
    intensity   = flops / bytes_moved
    return elapsed_s * 1e3, tflops, intensity


if device.type == "cuda":
    print(f"{'N':>6} | {'time (ms)':>10} | {'TFLOPs/s':>10} | {'FLOPs/byte':>12}")
    print("-" * 48)
    for N in (256, 512, 1024, 2048, 4096):
        ms, tflops, I = benchmark_gemm(N)
        print(f"{N:>6} | {ms:>10.3f} | {tflops:>10.2f} | {I:>12.1f}")
else:
    print("GEMM benchmark requires a CUDA device.")


**Observations.**

1. For small $N$, the kernel cannot saturate the device — there are not enough independent tiles to fill the SMs.
2. As $N$ grows, arithmetic intensity (which scales as $\mathcal{O}(N)$) crosses the machine balance point and throughput approaches the FP16 peak.
3. Tensor Cores require dimensions that are multiples of 8 (FP16) or 16 (INT8/FP8); ill-shaped matrices silently fall back to slower paths.

These are not academic curiosities. The choice between `hidden_dim = 768` and `hidden_dim = 760` can change measured throughput by 20 %.


## 6. NVIDIA Nsight: Beyond Operator-Level Profiling

`torch.profiler` is sufficient for operator-level diagnosis, but two questions remain beyond its reach:

- *Is a specific kernel achieving its theoretical occupancy?*
- *Where, exactly, is its time spent — instruction issue, memory dependency, branch divergence?*

Two complementary tools answer these questions:

### Nsight Systems (`nsys`)

A **system-wide tracer**. It captures CPU threads, CUDA API calls, kernel launches, NVTX ranges, and PCIe / NVLink traffic on a unified timeline. Use it to diagnose *coarse-grained* issues: CPU bottlenecks, dataloader stalls, kernel launch overhead, missed overlap between compute and communication.

Typical workflow on a SLURM cluster:

```bash
nsys profile \
    --trace=cuda,cudnn,nvtx,osrt \
    --output=trace_week1 \
    --capture-range=cudaProfilerApi \
    python scripts/train_step.py
```

The resulting `trace_week1.nsys-rep` opens in the Nsight Systems GUI.

### Nsight Compute (`ncu`)

A **kernel-level profiler**. It replays selected kernels and collects hardware performance counters — instructions issued, memory transactions, warp stalls — and presents them via the *Speed of Light* and *Memory Workload Analysis* sections.

```bash
ncu --set full \
    --target-processes all \
    -o ncu_report_week1 \
    python scripts/single_kernel.py
```

We will use both tools in the weeks ahead, particularly when assessing CUDA kernels we write ourselves (Week 2) and when diagnosing communication overhead in distributed training (Weeks 4–5).


## 7. Adding NVTX Ranges for Targeted Profiling

NVTX (NVIDIA Tools Extension) lets us mark regions of interest in our Python code. Nsight Systems will render these as named bands on its timeline, making it dramatically easier to locate the phenomenon under study.


In [ ]:
# NVTX-annotated forward pass. Run under `nsys profile` to see the bands.
try:
    import torch.cuda.nvtx as nvtx
except ImportError:
    nvtx = None

def annotated_step(model, x):
    if nvtx is not None and device.type == "cuda":
        nvtx.range_push("transformer_block")

    if nvtx is not None and device.type == "cuda":
        nvtx.range_push("attention")
    a, _ = model.attn(x, x, x, need_weights=False)
    if nvtx is not None and device.type == "cuda":
        nvtx.range_pop()

    x = model.ln1(x + a)

    if nvtx is not None and device.type == "cuda":
        nvtx.range_push("ffn")
    f = model.ffn(x)
    if nvtx is not None and device.type == "cuda":
        nvtx.range_pop()

    x = model.ln2(x + f)
    if nvtx is not None and device.type == "cuda":
        nvtx.range_pop()
    return x


with torch.no_grad():
    out = annotated_step(model, x)
print("Output shape:", tuple(out.shape))


## 8. Exercises

Complete the following before the next session. Solutions will be discussed in Week 2's recitation.

**Exercise 1.1 — Roofline classification.**  
For each of the kernels below, estimate its arithmetic intensity (in FLOPs/byte) assuming FP16 storage and FP32 accumulation. Classify each as compute- or memory-bound on the A100.

- Element-wise GELU on a tensor of shape `(B, L, D) = (32, 1024, 4096)`.
- Layer normalization on the same tensor.
- `nn.Linear(4096, 4096)` applied to the same tensor.

**Exercise 1.2 — Profiler walkthrough.**  
Modify the transformer block above to use `nn.functional.scaled_dot_product_attention`. Re-profile and report which operator now dominates the trace. Why is the picture different?

**Exercise 1.3 — Empirical roofline.**  
Extend `benchmark_gemm` to FP32 and BF16. Plot the measured throughput against the matrix size and compare to the theoretical peak in each precision. What fraction of peak does each precision achieve, and why?

**Exercise 1.4 — Nsight Systems (optional, requires Nsight installed).**  
Run the annotated step under `nsys profile` and locate the NVTX bands. Identify any gaps between consecutive CUDA kernel launches that exceed 50 µs. What might cause such gaps?


## 9. Summary

We began with the observation that deep learning workloads are governed by the *memory hierarchy* of modern GPUs, not by raw FLOP counts. The roofline model formalizes this intuition: kernels below the machine balance point are memory-bound and benefit primarily from algorithmic restructuring, while kernels above it are compute-bound and benefit from precision reductions and tensor-core utilization. We profiled a transformer block with `torch.profiler`, observed the dominance of GEMM in compute time and of element-wise operators in *latency*, and surveyed the Nsight family for deeper, hardware-counter-level analysis.

Week 2 turns from *measurement* to *intervention*: we will write our first CUDA kernel from scratch and progressively optimize it to approach the performance of cuBLAS.

## Further Reading

- Williams, S., Waterman, A., Patterson, D. (2009). *Roofline: An Insightful Visual Performance Model for Multicore Architectures.* Communications of the ACM, 52(4).
- NVIDIA. *CUDA C++ Programming Guide*, Chapter 5: Performance Guidelines.
- NVIDIA. *Nsight Systems Documentation* and *Nsight Compute Documentation*.
- Patterson, D., Hennessy, J. *Computer Architecture: A Quantitative Approach*, 6th ed., Chapter 4 (Data-Level Parallelism).
